# RAG PIPELINE TESTS

## SETTING PATHS

In [10]:


import os
import sys

# Get the absolute path of the parent root directory (project root)
#root_dir = os.path.abspath(os.path.join(".."))

# Append it to sys.path if it is not already there

root_dir = os.path.abspath(os.path.join("src"))
if root_dir not in sys.path:
    sys.path.append(root_dir)
print(root_dir)



c:\Users\Prime\OneDrive\Desktop\Nayatel_ai_assistant\Tests\src


In [26]:
from glob import glob
from pathlib import Path
import pymupdf as fitz

def infercategory(path: Path):
    return path.parent.name

pdf_dir = Path(os.path.join(root_dir, "..\\..\\backend\\data\\raw"))
pdf_paths = pdf_dir.glob('**/*.pdf')
loaded_files = []
categories = []
for path in pdf_paths:
    docs = fitz.open(path)
    loaded_files.append({"filename":path.name, "file_path":path})

print(len(loaded_files))
for file in loaded_files:
    if file["filename"] in categories:
        continue
    else:
        categories.append(file["filename"])

print(len(categories))
print(categories)


44
44
['5G in Pakistan.pdf', 'Benefits of fiber internet for homes and businesses.pdf', 'Best Internet Speed for Freelancers in Pakistan.pdf', 'Difference between 2.4 GHz and 5 GHz.pdf', 'Fiber vs DSL vs 4G_5G.pdf', 'How Many Devices Can Your Internet Plan Actually Support.pdf', 'How to Optimize Your Fiber Internet at Home.pdf', 'How to set up your fiber optic router.pdf', 'IPTV vs Cable TV.pdf', 'Nayatel Is Now Serving Enterprises Around the World.pdf', 'ONT Lights .pdf', 'What Is Fiber optic.pdf', 'Why Fiber Internet is Better for Gaming and Streaming.pdf', 'Contact us.pdf', 'discounts.pdf', 'faq.pdf', 'Hardware Changes.pdf', 'edge ont.pdf', 'hdmi cec settings.pdf', 'hik vision.pdf', 'Huawei GPON dual band ont user manaul.pdf', 'huawei gpon.pdf', 'huawei windows application for nwatch.pdf', 'naya box user manual.pdf', 'Ruijie.pdf', 'tplink router.pdf', 'ups manual.pdf', 'VOC.pdf', 'Wireless Air Remote Mouse.pdf', 'Payment options.pdf', 'Nayatel_Pricing_By_Location.pdf', 'Security Adv

## 1. (Does the vector DB actually have my data?)

In [2]:
import chromadb

client = chromadb.PersistentClient(config.VECTOR_DB_PATH)
collection = client.get_collection("nayatel_docs")
print(collection.count())

115


### RESULTS:
- EXPECTED OUTPUT 115
- OBTAINED OUTPUT 115


## 2. Raw retrieval scores — get real numbers before guessing thresholds

In [3]:
retriever = Retriver()

in_scope = retriever.retrive("What internet packages are available?", top_k=5)
out_scope = retriever.retrive("Does NayaTel offer Starlink packages?", top_k=5)

for label, hits in [("IN-SCOPE", in_scope), ("OUT-OF-SCOPE", out_scope)]:
    print(f"\n--- {label} ---")
    for h in hits:
        print(f"{h['score']:.4f}  {h['source']} p{h['page']}  {h['text'][:80]}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


--- IN-SCOPE ---
1.0414  Fiber vs DSL vs 4G_5G.pdf p1  Fiber vs. DSL vs. 4G/5G: Which Internet Is Best? 6 months ago Feb 3, 26 Internet
1.0786  Unlimited_bundle.pdf p1  Maximize your social media with Unlimited Bundles Get up to 75 Mbps dedicated sp
1.0835  Benefits of fiber internet for homes and businesses.pdf p1  Benefits of fiber internet for homes and businesses 6 months ago Jan 13, 26 Reli
1.0946  Nayatel_Pricing_By_Location.pdf p1  Nayatel Internet Pricing by Location Effective from 2026-07-01 | Compiled 2026-0
1.1206  speed_up.pdf p1  Boost your internet Speed up to 500 Mbps Increase your bandwidth with unlimited 

--- OUT-OF-SCOPE ---
0.9333  Nayatel_Pricing_By_Location.pdf p14  " page or approach our sales team at 1441. Can I upgrade my package after signin
0.9409  Nayatel_Pricing_By_Location.pdf p13  7,200 Unlimited (FUP applies) 200 Mbps Rs. 14,000 Unlimited (FUP applies) 350 Mb
0.9815  faq.pdf p1  1.About Nayatel? Nayatel is the leading fiber internet and IT service provi

In [4]:
found = retriever.retrive("test query", top_k=1)
print(found[0].keys())

dict_keys(['ids', 'text', 'source', 'page', 'category', 'chunk_index', 'score'])


## 3. Full Phase 5 test — your 10 questions, checked for relevance

In [5]:
test_questions = [
    "What packages are available?",
    "How can I get a new connection?",
    "How long does installation take?",
    "How do I pay my bill?",
    "What should I do if my internet isn't working?",
    "How can I contact support?",
    "What documents are required?",
    "Can I upgrade my package?",
    "What happens if I don't pay my bill?",
    "What services does NayaTel provide?",
]

for q in test_questions:
    hits = retriever.retrive(q, top_k=3)
    print(f"\nQ: {q}")
    for h in hits:
        print(f"  [{h['category']}] {h['source']} — {h['text'][:60]}")


Q: What packages are available?
  [internet] Unlimited_bundle.pdf — Maximize your social media with Unlimited Bundles Get up to 
  [blog] Why Fiber Internet is Better for Gaming and Streaming.pdf — . Not mainstream yet, but coming. - VR/AR gaming: High-quali
  [blog] What Is Fiber optic.pdf — - Ultra-fast speeds for streaming, downloads, and cloud-base

Q: How can I get a new connection?
  [manuals] Ruijie.pdf — Ruijie RG-EW1200 User Guide Router Specifications 1.1 Front 
  [blog] How to Optimize Your Fiber Internet at Home.pdf — To change it, open your router admin panel, find the WAN or 
  [manuals] tplink router.pdf — Dual-Band TP-LINK Router User Guide Router Specifications TP

Q: How long does installation take?
  [blog] What Is Fiber optic.pdf — video calls and cloud work. - Low-latency network design: En
  [blog] How to set up your fiber optic router.pdf — -speed WAN connections. Can fiber be installed in apartments
  [video] cabel_tv.pdf — Upgrade your Cable TV with fiber Enjo

## 4. Category-specific stress tests

In [6]:
category_specific_questions = [
    "How do I reset my ONT?",
    "What is the difference between 2.4Ghz and 5Ghz?",
    "How do i change my wifi password?"
]

for q in category_specific_questions:
    hits = retriever.retrive(q, top_k=3)
    print(f"\nQ: {q}")
    for h in hits:
        print(f"  [{h['category']}] {h['source']} — {h['text'][:60]}")


Q: How do I reset my ONT?
  [blog] ONT Lights .pdf — supply During load shedding, the ONT will switch off unless 
  [blog] ONT Lights .pdf — ISP’s voice service configuration. What Should Normal ONT Li
  [blog] ONT Lights .pdf — What Do the Lights on Your ONT Mean? Complete ONT Light Guid

Q: What is the difference between 2.4Ghz and 5Ghz?
  [blog] Difference between 2.4 GHz and 5 GHz.pdf — 2.4 GHz vs 5 GHz WiFi: What Is the Difference and Which Shou
  [blog] Difference between 2.4 GHz and 5 GHz.pdf — If your router is dual-band, as most modern routers are, bot
  [manuals] tplink router.pdf — to Wireless 2.4GHz >> Basic Settings. Update standard parame

Q: How do i change my wifi password?
  [manuals] Ruijie.pdf — Ruijie RG-EW1200 User Guide Router Specifications 1.1 Front 
  [manuals] tplink router.pdf — Dual-Band TP-LINK Router User Guide Router Specifications TP
  [manuals] tplink router.pdf — to Wireless 2.4GHz >> Basic Settings. Update standard parame


## 5. Full pipeline test — answer() end-to-end

In [7]:
from pipeline import RagPipelines
from history import History
from client import LLMClient   # adjust import path to match your actual structure

# 1. build the dependencies
retriever = Retriver()
llm_client = LLMClient()
history = History()
config.ENV_PATH

rag_pipeline = RagPipelines(
    retriver=retriever,
    llm_client=llm_client,
    history=history,
    similarity_threshold = config.SIMILARITY_THRESHOLD,
    top_k = config.TOP_K
)
result = rag_pipeline.answer("What are prices of internet packages is islamabad?", session_id="test1")
print(result)

result2 = rag_pipeline.answer("Does NayaTel offer Starlink packages?", session_id="test1")
print(result2)  

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

{'answer': 'Here are the Islamabad prices from the provided list (all PKR/month + tax; Unlimited — FUP applies):\n\nUnlimited Internet only:\n- 30 Mbps — Rs. 2,225\n- 40 Mbps — Rs. 3,450\n- 50 Mbps — Rs. 4,300\n- 70 Mbps — Rs. 5,300\n- 100 Mbps — Rs. 7,200\n- 200 Mbps — Rs. 14,000\n- 350 Mbps — Rs. 24,000\n\nTriple Play (Internet + Cable TV + Phone):\n- 30 Mbps — Rs. 2,525\n- 40 Mbps — Rs. 3,750\n- 50 Mbps — Rs. 4,600\n- 70 Mbps — Rs. 5,600\n- 100 Mbps — Rs. 7,500\n- 200 Mbps — Rs. 14,300\n- 350 Mbps — Rs. 24,300\n\nIf you need help choosing a plan or want installation/contact details, I do not have this information currently please contact customer service helpline.', 'source': [{'title': 'Nayatel_Pricing_By_Location.pdf', 'page': 4}, {'title': 'Nayatel_Pricing_By_Location.pdf', 'page': 1}, {'title': 'Nayatel_Pricing_By_Location.pdf', 'page': 5}, {'title': 'Nayatel_Pricing_By_Location.pdf', 'page': 10}, {'title': 'Nayatel_Pricing_By_Location.pdf', 'page': 7}]}
{'answer': 'I AM UNABLE 

## 6. Conversation history / follow-up test

In [8]:
r1 = rag_pipeline.answer("Tell me about the 20mbps package", session_id="test2")
r2 = rag_pipeline.answer("How much does it cost?", session_id="test2")  # no "20mbps" in this query
print(r2)

{'answer': '20 Mbps Unlimited (Internet-only): Rs. 1,775/month + tax (listed for cities like Peshawar, Rawalpindi, Muzaffargarh).\n\n20 Mbps Triple Play (Internet + Cable TV + Phone): Rs. 2,075/month + tax.\n\nAvailability and exact pricing can vary by city (e.g., Islamabad’s entry tier starts at 30 Mbps). If you want confirmation for your specific city or address, I DO NOT HAVE THIS INFORMATION CURRENTLY PLEASE CONTACT CUSTOMER SERVICE HELPLINE.', 'source': [{'title': 'speed_up.pdf', 'page': 1}, {'title': 'Unlimited_bundle.pdf', 'page': 1}, {'title': 'Nayatel_Pricing_By_Location.pdf', 'page': 4}, {'title': 'Nayatel_Pricing_By_Location.pdf', 'page': 7}, {'title': 'Nayatel_Pricing_By_Location.pdf', 'page': 5}]}


## 7. Error handling checks

In [ ]:
rag_pipeline.answer("", session_id="test3")

{'answer': 'please enter a question!!!', 'source': []}

## 8. Chunking strategy comparison (place holder)

In [ ]:
# NOTE PLACE HOLDER FOR FUTURE DIFFERENT CHUNKING STRATEGY TESTS AS ALREADY IMPLEMENTED